In [ ]:
# @title Plotting utils and load particle data
import re

import matplotlib as mpl
from matplotlib import rcParams
from matplotlib.gridspec import GridSpec
import matplotlib.image as img
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
import tensorflow as tf
import xarray_tensorstore
import xarray


rcParams['text.usetex'] = False

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

div_colors = (
    np.flipud(
        np.array([
            [103, 0, 31],
            [178, 24, 43],
            [214, 96, 77],
            [244, 165, 130],
            [253, 219, 199],
            [247, 247, 247],
            [209, 229, 240],
            [146, 197, 222],
            [67, 147, 195],
            [33, 102, 172],
            [5, 48, 97],
        ])
    )
    / 255
)
div = mpl.colors.LinearSegmentedColormap.from_list('div', div_colors)


seq_colors = (
    np.flipud(
        np.array([
            [255, 247, 236],
            [254, 232, 200],
            [253, 212, 158],
            [253, 187, 132],
            [252, 141, 89],
            [239, 101, 72],
            [215, 48, 31],
            [179, 0, 0],
            [127, 0, 0],
        ])
    )
    / 255
)
seq = mpl.colors.LinearSegmentedColormap.from_list('seq', seq_colors)

nat_colors = [
    '#e64b35',
    # '#4dbbd5',
    '#00a087',
    '#3c5488',
    '#f39b7f',
    '#8491b4',
    '#91d1c2',
    '#dc0000',
    '#7e6148',
    '#b09c85',
]

tmp_colors = ['navy', 'indigo', 'lightsteelblue', 'thistle']

data_path = ''  #@param{type: 'string'}
plot_path = f'{data_path}/figures'
write_to_disk = True  # @param{type: 'boolean'}

labels = {
    'u1_mc0': 'Baseline',
    'u0.333_mc0': '1/3 Wind',
    'u0.666_mc0': '2/3 Wind',
    'u1_mc0.3': '30% MC',
}

sims = {
    'Baseline': xarray.open_zarr(
        f'{data_path}/u1_mc0/output.zarr', decode_timedelta=True
    ),
    '1/3 Wind': xarray.open_zarr(
        f'{data_path}/u0.333_mc0/output.zarr', decode_timedelta=True
    ),
    '2/3 Wind': xarray.open_zarr(
        f'{data_path}/u0.666_mc0/output.zarr', decode_timedelta=True
    ),
    '30% MC': xarray.open_zarr(
        f'{data_path}/u1_mc0.3/output.zarr', decode_timedelta=True
    ),
}

def load_particle_postprocess(mask_pattern, as_dict=True):
  sims = ['u1_mc0', 'u0.333_mc0', 'u0.666_mc0', 'u1_mc0.3']
  out = {}
  for sim in sims:
    for fn in tf.gfile.Glob(f'{data_path}/{sim}/postprocess/particles/{mask_pattern}'):
      m = re.search(r'h2o/(.*)/postprocess.*particles/(.*?)_', fn)
      t = int(m.group(2))
      ds = xarray_tensorstore.open_zarr(fn)
      ds = ds.assign_coords(t=np.timedelta64(t, 'm'))
      out.setdefault(m.group(1), []).append((t, xarray_tensorstore.read(ds)))
  if as_dict:
    return {k: xarray.concat([t_ds[1] for t_ds in sorted(v)], dim='t') for k, v in out.items()}
  ds_out = [
      xarray.concat(
          [t_ds[1] for t_ds in sorted(v)], dim='t'
      ).assign_coords(sim=k)
      for k, v in out.items()
  ]
  return xarray.concat(ds_out, dim='sim')


def summarize_masks(d):
  out = {}
  for k in d:
    out[k] = d[k].masks.sum('p')
  return out

parcels = load_particle_postprocess('*budget_20250527b/20250609c.zarr')
parcels_budget = load_particle_postprocess('*budget_20250602_30_50_1M/20250603b.zarr', as_dict=False)


!mkdir -p {plot_path}

In [ ]:
# @title Plotting functions.
ENERGY_VARS = {
    R'energy_budget_potential_energy': '$\mathcal{W}_b$',
    R'energy_budget_latent_heat_fuel_moisture': '$\mathcal{E}_{evap}$',
    R'energy_budget_combustion': '$\mathcal{E}_{comb}$',
    R'energy_budget_drag_heating': '$\mathcal{W}_d$',
    R'energy_budget_radiation': '$\mathcal{E}_{rad}$',
    R'energy_budget_latent_heat_microphysics': '$\mathcal{E}_{ph-tr}$',
    R'energy_budget_mass_source': '$\mathcal{T}_{ph-exch}$',
}


WATER_VARS = {
    'water_budget_dehydration': '$\mathcal{M}_{evap}$',
    'water_budget_combustion': '$\mathcal{M}_{comb}$',
    'water_budget_rain': '$\mathcal{M}_{rain}$',
    'water_budget_snow': '$\mathcal{M}_{snow}$',
    'water_budget_mass_source': '$\mathcal{M}_{ma-exch}$',
}


def simple_range(x0, x1):
  x = np.asarray([x0, x1])
  exp = np.max(np.floor(np.log10(np.abs(x) + 1e-12)))
  first_digits = np.floor(x / 10**exp)
  return first_digits[0] * 10**exp, (first_digits[1] + 1) * 10**exp


def plot_bar(budgets, rs, ax, **kwargs):
  bar_width = kwargs.get('bar_width', 0.18)

  for i, (sim, bud) in enumerate(budgets.items()):
    ax.bar(
        rs[i], bud.values(), color=colors[i], width=bar_width, label=labels[sim]
    )
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.tick_params(labeltop=False)  # Hide x-axis labels on upper subplot

  ax.set_ylim(kwargs['ylim'])  # Upper limit for the broken axis
  ax.set_ylabel(kwargs['ylabel'], fontsize=10)
  ax.set_yticks(kwargs['yticks'])
  ax.set_xticks(
      [r + bar_width for r in range(len(kwargs['categories']))],
      kwargs['categories'],
  )

  plt.tight_layout()  # Adjust layout to prevent labels from overlapping

  ax.text(**kwargs['text_kwargs'])


def plot_bar_on_broken_axes(budgets, rs, axes, **kwargs):
  ax1, ax2, ax3 = axes
  bar_width = 0.18

  for i, (sim, bud) in enumerate(budgets.items()):
    ax1.bar(
        rs[i], bud.values(), color=colors[i], width=bar_width, label=labels[sim]
    )
    ax1.set_ylim(kwargs['top_ylim'])  # Upper limit for the broken axis
    ax1.spines['bottom'].set_visible(False)
    ax1.tick_params(labelbottom=False)  # Hide x-axis labels on upper subplot
    ax1.set_xticks([])
    ax1.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

    ax2.bar(
        rs[i], bud.values(), color=colors[i], width=bar_width, label=labels[sim]
    )
    ax2.set_ylim(kwargs['middle_ylim'])  # Lower limit for the broken axis
    ax2.spines['top'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.tick_params(
        labeltop=False, labelbottom=False
    )  # Hide x-axis labels on upper subplot
    ax2.set_xticks([])
    ax2.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

    ax3.bar(
        rs[i], bud.values(), color=colors[i], width=bar_width, label=labels[sim]
    )
    ax3.set_ylim(kwargs['bottom_ylim'])  # Lower limit for the broken axis
    ax3.spines['top'].set_visible(False)
    ax3.tick_params(labeltop=False)  # Hide x-axis labels on upper subplot
    ax3.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

  # Add break indicators (optional, more complex for precise placement)
  d = 0.006  # how big to make the diagonal lines in axes coordinates
  kwargs1 = dict(transform=ax1.transAxes, color='k', clip_on=False)
  ax1.plot((-d, +d), (-d, +d), **kwargs1)  # top-left diagonal
  ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs1)  # top-right diagonal

  kwargs1.update(transform=ax2.transAxes)  # switch to the bottom axes
  ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs1)  # bottom-left diagonal
  ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs1)  # bottom-right diagonal
  ax2.plot((-d, +d), (-d, +d), **kwargs1)  # top-left diagonal
  ax2.plot((1 - d, 1 + d), (-d, +d), **kwargs1)  # top-right diagonal

  kwargs1.update(transform=ax3.transAxes)  # switch to the bottom axes
  ax3.plot((-d, +d), (1 - d, 1 + d), **kwargs1)  # bottom-left diagonal
  ax3.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs1)  # bottom-right diagonal

  ax2.set_ylabel(kwargs['ylabel'], fontsize=10)
  ax3.set_xticks(
      [r + bar_width for r in range(len(kwargs['categories']))],
      kwargs['categories'],
  )
  plt.tight_layout()  # Adjust layout to prevent labels from overlapping

  ax1.text(**kwargs['text_kwargs'])


def plot_on_brokenaxes(x, y, ax1, ax2, **kwargs):
  fontsize = kwargs.get('fontsize', 10)

  # plot the same data on both Axes
  plt_args = {k: kwargs.get(k, None) for k in ['color', 'linestyle', 'label']}
  ax1.plot(x, y, **plt_args)
  ax2.plot(x, y, **plt_args)

  # zoom-in / limit the view to different portions of the data
  ax1.set_ylim(kwargs['ylim_top'])
  ax1.set_yticks(kwargs['yticks_top'])
  ax2.set_ylim(kwargs['ylim_bottom'])
  ax2.set_yticks(kwargs['yticks_bottom'])

  ax1.text(**kwargs['text_kwargs'])

  if 'xlim' not in kwargs:
    x_min, x_max = x.quantile([0, 1], dim=['sim', 'z']).data
    x_margin = (x_max - x_min) * 0.03
    if x_margin == 0:
      x_margin = 0.1
    x_min -= x_margin
    x_max += x_margin

    xlim = simple_range(x_min, x_max)
  else:
    xlim = kwargs['xlim']

  ax1.set_xlim(xlim)
  ax2.set_xlim(xlim)
  xaxis_sci = kwargs.get('xaxis_sci', True)
  if xaxis_sci:
    ax2.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))

  if kwargs.get('show_ylabel', 'ylabel' in kwargs):
    ax1.set_ylabel(kwargs['ylabel'], fontsize=fontsize, labelpad=16)
    ax1.yaxis.label.set_position((-10, 0))
    ax1.tick_params(axis='y', labelsize=fontsize)
    ax2.tick_params(axis='y', labelsize=fontsize)
  else:
    ax1.set_yticklabels([])
    ax2.set_yticklabels([])

  if kwargs.get('show_xlabel', 'xlabel' in kwargs):
    ax2.set_xlabel(kwargs['xlabel'], fontsize=fontsize)
    ax2.xaxis.get_offset_text().set_fontsize(fontsize / 3 * 2)
    ax2.tick_params(axis='x', labelsize=fontsize)
  else:
    ax2.set_xticklabels([])

  # hide the spines between ax and ax2
  ax1.spines.bottom.set_visible(False)
  ax2.spines.top.set_visible(False)
  ax1.xaxis.tick_top()
  ax1.tick_params(top=False)
  ax1.tick_params(labeltop=False)  # don't put tick labels at the top
  ax2.xaxis.tick_bottom()

  # Now, let's turn towards the cut-out slanted lines.
  # We create line objects in axes coordinates, in which (0,0), (0,1),
  # (1,0), and (1,1) are the four corners of the Axes.
  # The slanted lines themselves are markers at those locations, such that the
  # lines keep their angle and position, independent of the Axes size or scale
  # Finally, we need to disable clipping.

  d = 0.25  # proportion of vertical to horizontal extent of the slanted line
  axes_args = dict(
      marker=[(-1, -d), (1, d)],
      markersize=12,
      linestyle='none',
      color='k',
      mec='k',
      mew=1,
      clip_on=False,
  )
  ax1.plot([0, 1], [0, 0], transform=ax1.transAxes, **axes_args)
  ax2.plot([0, 1], [1, 1], transform=ax2.transAxes, **axes_args)


def plot_xy(
    ax,
    subfig_label,
    sim_name='u0.666_mc0',
    t=27,
    height=12,
    show_colorbar=True,
    show_xlabel=True,
    show_ylabel=True,
):
  sim_dir = f'{data_path}/{sim_name}'

  sim = (
      xarray.open_zarr(f'{sim_dir}/output.zarr', decode_timedelta=True)
      .sel(t=np.timedelta64(t, 'm'))
      .sel(x=slice(0, 1.5e4), y=slice(-4e3, 4e3))
      .sel(z=height, method='nearest')
  )
  u = sim.u.T
  v = sim.v.T
  w = sim.w.T
  t_s = sim.T_s.T
  q_r = sim.q_r.T
  u_mag = np.sqrt(u**2 + v**2 + w**2)

  x = sim.x * 1e-3
  y = sim.y * 1e-3

  xx = np.linspace(0, 2e4, 401)
  yy = np.linspace(-4e3, 4e3, 301)
  uu = u.interp(x=xx, y=yy)
  vv = v.interp(x=xx, y=yy)

  c = ax.pcolormesh(
      x,
      y,
      u_mag,
      vmin=0,
      vmax=20,
      cmap=mpl.colormaps['viridis'],
      shading='gouraud',
  )
  ax.contour(
      x,
      y,
      t_s,
      levels=[600.0],
      colors='r',
      linewidths=0.4,
  )
  ax.contour(
      x,
      y,
      q_r,
      levels=[1e-5],
      colors='cyan',
      linewidths=0.4,
  )
  ax.streamplot(
      xx * 1e-3,
      yy * 1e-3,
      uu,
      vv,
      density=1,
      linewidth=0.4,
      arrowsize=0.4,
      color='w',
  )

  if show_colorbar:
    cb = plt.colorbar(c, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(r'$|\boldsymbol{u}|$ (m/s)', size=10)
    cb.ax.tick_params(labelsize=10)

  ax.set_xlim([0, 15])
  ax.set_xticks([0, 5, 10, 15])
  if show_xlabel:
    ax.set_xlabel('x (km)', fontsize=10)
    ax.set_xticklabels([0, 5, 10, 15])
    ax.tick_params(axis='x', labelsize=10)
  else:
    ax.set_xticklabels([])

  ax.set_ylim([-4, 4])
  ax.set_yticks([-4, -2, 0, 2, 4])
  if show_ylabel:
    ax.set_ylabel('y (km)', fontsize=10)
    ax.set_yticklabels([-4, -2, 0, 2, 4])
    ax.tick_params(axis='y', labelsize=10)
  else:
    ax.set_yticklabels([])

  ax.set_aspect('equal')

  ax.text(1, 2, subfig_label, fontsize=10, fontweight='bold', color='w')

  return c


def imread(filepath):
  with gfile.GFile(filepath, 'rb') as f:
    pil_image = Image.open(f)
    return np.array(pil_image)

In [ ]:
#@title Figure 1

filename = f'{plot_path}/demo/pyrocb_demo.png'
pyrocb = imread(filename)

filename = f'{plot_path}/demo/fire_whirl.png'
fire = imread(filename)

filename = f'{plot_path}/demo/plume_with_parcels.png'
plume = imread(filename)

fig = plt.figure(figsize=(7.24, 9.65))
gs = GridSpec(2, 2, figure=fig) # 2 rows, 2 columns
gs.update(wspace=0.025, hspace=0.01)

# Create subplots using GridSpec
ax1 = fig.add_subplot(gs[0, :]) # Spans both columns in the first row
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Plot on each subplot
ax1.imshow(pyrocb[100:-40, 125:-300, :])
ax1.set_xticks([])
ax1.set_yticks([])
for spine in ax1.spines.values():
  spine.set_edgecolor('none')
ax1.text(10, 50, 'A', fontsize=10, fontweight='bold')

ax2.imshow(fire[220:, 500:-500, :])
ax2.set_xticks([])
ax2.set_yticks([])
for spine in ax2.spines.values():
  spine.set_edgecolor('none')
ax2.text(10, -20, 'B', fontsize=10, fontweight='bold')

ax3.imshow(plume)
ax3.set_xticks([])
ax3.set_yticks([])
for spine in ax3.spines.values():
  spine.set_edgecolor('none')
ax3.text(10, -10, 'C', fontsize=10, fontweight='bold')

if write_to_disk:
  filename = f'{plot_path}/fig_1.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=600, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure 2
sim_name = 'u1_mc0'
t = 90

q_labels = {
    'q_c': '$q_c$ (g kg$^{-1}$)',
    'q_r': '$q_r$ (g kg$^{-1}$)',
    'q_t': '$q_t$, $q_c$ (g kg$^{-1}$)',
    'w': '$w$ (m s$^{-1}$)',
}
sim_dir = f'{data_path}/{sim_name}'
cmaps = ['binary', 'Blues', div]

Z_BL = 4e3  # m

sim = (
    xarray.open_zarr(f'{sim_dir}/output.zarr', decode_timedelta=True)
    .sel(t=np.timedelta64(t, 'm'))
    .sel(x=slice(0, 2e4), y=slice(-1e2, 1e2), z=slice(0, 1.5e4))
)

filename = f'{plot_path}/demo/block_flow_rounded.png'
with gfile.GFile(filename, 'rb') as f:
  mech = img.imread(f)

filename = f'{plot_path}/demo/baseline_parcel_count.png'
with gfile.GFile(filename, 'rb') as f:
  flow = img.imread(f)


def plot_xz(ax, varnames, subfig_label):
  for i, varname in enumerate(varnames):
    val = sim[varname].sel(y=0, method='nearest').T
    if varname in ['q_c', 'q_r', 'q_t']:
      val = val * 1e3
      vmin = 0
      vmax = 8
      cmap = cmaps[i]
    elif varname == 'w':
      vmin = -20
      vmax = 60
      cmap = 'viridis'
    else:
      raise ValueError(f'Unknown variable: {varname}')
    c = ax.pcolormesh(
        sim.x * 1e-3,
        sim.z * 1e-3,
        val,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        alpha=1 - i / len(varnames),
    )
    if varname == 'q_c':
      continue
    cb = fig.colorbar(
        c,
        ax=ax,
        fraction=0.046,
        pad=0.1,
        location='top',
        extend='both' if varname == 'w' else 'max',
    )
    cb.set_label(q_labels[varname], size=10)
    cb.ax.tick_params(labelsize=10)
    cb.ax.locator_params(nbins=4)

  ax.set_xlim([0, 20])
  ax.set_aspect('equal')
  ax.set_xlabel('x (km)', fontsize=10)
  ax.set_yticks([0, 5, 10, 15, 20])
  ax.tick_params(axis='x', labelsize=10)

  ax.set_ylim([0, 15])
  ax.set_yticks([0, 5, 10, 15])
  ax.set_ylabel('z (km)', fontsize=10)
  ax.tick_params(axis='y', labelsize=10)
  # if 'q_t' in varnames:
  #   ax.set_ylabel('z (km)', fontsize=10)
  #   ax.tick_params(axis='y', labelsize=10)
  # else:
  #   ax.set_ylabel(None)
  #   ax.set_yticklabels([])

  ax.text(
      2,
      12,
      subfig_label,
      fontsize=10,
      fontweight='bold',
      color='w' if subfig_label == 'C' else 'k',
  )
  if subfig_label == 'A':
    ax.text(4.5, 12, '$q_t$', fontsize=10)
  elif subfig_label == 'B':
    ax.text(4.5, 12, '$q_c$, $q_r$', fontsize=10)
  elif subfig_label == 'C':
    ax.text(4.5, 12, '$w$', fontsize=10, color='w')


varnames_group = (
    [
        'q_t',
    ],
    [
        'q_c',
        'q_r',
    ],
    [
        'w',
    ],
)

fig = plt.figure(figsize=(7.24, 3.62), constrained_layout=False)
gs1 = fig.add_gridspec(
    1, 3, bottom=0.66, top=0.98, width_ratios=[1, 1, 1]
)  # wspace=0.025, hspace=0.01)
gs2 = fig.add_gridspec(
    1,
    2,
    bottom=0.02,
    top=0.56,
    width_ratios=[3, 2],
)  # wspace=0.025, hspace=0.01)

ax1 = fig.add_subplot(gs1[0, 0])
ax2 = fig.add_subplot(gs1[0, 1])
ax3 = fig.add_subplot(gs1[0, 2])
ax4 = fig.add_subplot(gs2[0, 0])
ax5 = fig.add_subplot(gs2[0, 1])

subfig_labels = ('A', 'B', 'C')
for i, ax in enumerate([ax1, ax2, ax3]):
  plot_xz(ax, varnames_group[i], subfig_labels[i])

ax4.imshow(mech)
ax4.set_xticks([])
ax4.set_yticks([])
for spine in ax4.spines.values():
  spine.set_edgecolor('none')
ax4.text(10, 40, 'D', fontsize=10, fontweight='bold')

ax5.imshow(flow)
ax5.set_xticks([])
ax5.set_yticks([])
for spine in ax5.spines.values():
  spine.set_edgecolor('none')
ax5.text(10, 20, 'E', fontsize=10, fontweight='bold')

# fig.set_layout_engine('tight')

if write_to_disk:
  filename = f'{plot_path}/fig_2.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure 3


def gaussian_filter(da, sigma):
  return xarray.apply_ufunc(
      ndimage.gaussian_filter, da, kwargs={'sigma': sigma}, keep_attrs=True
  )


def prepare_data(data_path):
  filepaths = {
      'Baseline': f'{data_path}/u1_mc0/postprocess',
      '1/3 Wind': f'{data_path}/u0.333_mc0/postprocess',
      '2/3 Wind': f'{data_path}/u0.666_mc0/postprocess',
      '30% MC': f'{data_path}/u1_mc0.3/postprocess',
  }

  sims = {}
  for k, filepath in filepaths.items():
    buf = xarray.open_zarr(
        f'{filepath}/flux_and_entrainment.zarr',
        decode_timedelta=True,
    )
    sims[k] = buf.isel(t=~buf.t.isnull())

  t_min = 20  # minutes
  z_max = 1.5e4  # meter
  return {
      k: (
          agg_ds.flux.sel(
              t=slice(np.timedelta64(t_min, 'm'), None), z=slice(0, z_max)
          )
          .mean(['t'])
          .compute()
      )
      for k, agg_ds in sims.items()
  }


def add_sum(pds):
  src_sum = 0
  for v in pds['var'].data:
    if v.startswith('energy_budget'):
      src_sum += pds.sel(var=v)
  rad_total = pds.sel(var='energy_budget_radiation') + pds.sel(
      var='energy_budget_solid_radiation'
  )
  return xarray.concat(
      [
          pds,
          src_sum.assign_coords(var='energy_budget_total'),
          rad_total.assign_coords(var='energy_budget_total_radiation'),
      ],
      dim='var',
  )


parcel_plot = (
    parcels_budget.sel(w_dir='up', mask='plume')
    # .sel(start_min=slice(np.timedelta64(35, 'm'), np.timedelta64(45, 'm')))
    .sel(t=slice(np.timedelta64(35, 'm'), None)).mean('t')
    / 1e3
)
parcel_plot = add_sum(parcel_plot)

flux_mean = prepare_data(data_path)

bottom = 0.02
left = 0.02
height = 0.41
width = 0.25
hspace = 0.16
wspace = 0.115

fig = plt.figure(figsize=(7.24, 3), constrained_layout=False)

# The flux.
gs1 = fig.add_gridspec(
    1,
    1,
    bottom=bottom + height + hspace,
    top=bottom + 2 * height + hspace,
    left=left,
    right=left + width,
)
ax1 = fig.add_subplot(gs1[0, 0])
for i, (k, f) in enumerate(flux_mean.items()):
  ax1.plot(f, f.z / 1e3, label=k, color=colors[i])

ax1.set_xlabel('Mass Flux (kg s$^{-1}$)', fontsize=10)
ax1.set_xlim([0, 4e8])
ax1.xaxis.get_offset_text().set_fontsize(7)
ax1.tick_params(axis='x', labelsize=10)

ax1.set_ylabel('z (km)', fontsize=10)
ax1.set_ylim([0, 15])
ax1.tick_params(axis='y', labelsize=10)

ax1.text(x=0.3e8, y=13, s='A', fontsize=10, fontweight='bold')

axes = []
for r in range(2):
  for c in range(3):
    if r == 0 and c == 0:
      continue
    gs = fig.add_gridspec(
        2,
        1,
        bottom=bottom + (1 - r) * (height + hspace),
        top=bottom + (1 - r) * (height + hspace) + height,
        left=left + c * (width + wspace),
        right=left + c * (width + wspace) + width,
        height_ratios=[1, 1],
        hspace=0.04,
    )
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    axes.append([ax1, ax2])

# Combustion energy.
v = 'energy_budget_combustion'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[2][0],
      axes[2][1],
      xlim=[-10, 20],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-8, y=9, s='D', fontsize=10, fontweight='bold')
  )

# Potential energy.
v = 'energy_budget_potential_energy'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[0][0],
      axes[0][1],
      xlim=[-2e-2, 1e-2],
      ylim_top=[0.5, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-2, y=9, s='B', fontsize=10, fontweight='bold')
  )

# Evaporation energy.
v = 'energy_budget_latent_heat_fuel_moisture'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[1][0],
      axes[1][1],
      xlim=[-2e-1, 1e-1],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-1, y=9, s='C', fontsize=10, fontweight='bold')
  )

# Evaporation moisture.
v = 'water_budget_dehydration'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'] * 1e3,
      pds_plot.z * 1e-3,
      axes[3][0],
      axes[3][1],
      xlim=[-1e-5, 4e-5],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-0.8e-5, y=9, s='E', fontsize=10, fontweight='bold')
  )

# Combustion moisture.
v = 'water_budget_combustion'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'] * 1e3,
      pds_plot.z * 1e-3,
      axes[4][0],
      axes[4][1],
      xlim=[-1e-4, 4e-4],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-0.8e-4, y=9, s='F', fontsize=10, fontweight='bold')
  )

if write_to_disk:
  filename = f'{plot_path}/fig_3.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure 3 (bar chart)
def add_sum(pds):
  src_sum = 0
  for v in pds['var'].data:
    if v.startswith('energy_budget'):
      src_sum += pds.sel(var=v)
  rad_total = pds.sel(var='energy_budget_radiation') + pds.sel(
      var='energy_budget_solid_radiation'
  )
  return xarray.concat(
      [
          pds,
          src_sum.assign_coords(var='energy_budget_total'),
          rad_total.assign_coords(var='energy_budget_total_radiation'),
      ],
      dim='var',
  )


parcel_plot = (
    parcels_budget.sel(w_dir='up', mask='plume')
    # .sel(start_min=slice(np.timedelta64(35, 'm'), np.timedelta64(45, 'm')))
    .sel(t=slice(np.timedelta64(35, 'm'), None)).mean('t')
)
parcel_plot = add_sum(parcel_plot)

bottom = 0.02
left = 0.02
height = 0.41
width = 0.25
hspace = 0.16
wspace = 0.115
bar_width = 0.18

fig = plt.figure(figsize=(7.24, 4.7), constrained_layout=False)

# The flux.
gs1 = fig.add_gridspec(
    1,
    1,
    bottom=bottom + height + hspace,
    top=bottom + 2 * height + hspace,
    left=left,
    right=left + width,
)
ax1 = fig.add_subplot(gs1[0, 0])
for i, (k, f) in enumerate(flux_mean.items()):
  ax1.plot(f, f.z / 1e3, label=k, color=colors[i])

ax1.set_xlabel('Mass Flux (kg s$^{-1}$)', fontsize=10)
ax1.set_xlim([0, 4e8])
ax1.xaxis.get_offset_text().set_fontsize(7)
ax1.tick_params(axis='x', labelsize=10)

ax1.set_ylabel('z (km)', fontsize=10)
ax1.set_ylim([0, 15])
ax1.tick_params(axis='y', labelsize=10)

ax1.text(x=0.3e8, y=13, s='A', fontsize=10, fontweight='bold')

# The moisture budget.
budgets = {}
for sim in labels.keys():
  buf = {}
  for v, l in WATER_VARS.items():
    buf[l] = (
        parcel_plot.sel(sim=sim, var=v)
        .sel(z=slice(0, 10e3))['mean']
        .integrate(coord='z')
    )
  budgets[sim] = buf

rs = [np.arange(len(WATER_VARS))]
for i in range(1, len(labels)):
  rs.append(rs[i - 1] + bar_width)

# gs = fig.add_gridspec(
#     3,
#     1,
#     height_ratios=[1, 1, 1],
#     top=bottom + 2 * height + hspace,
#     bottom=bottom + height + hspace,
#     left=left + width + wspace,
#     right=left + 3 * width + 2 * wspace,
#     hspace=0.06,
# )
# ax1 = fig.add_subplot(gs[0, 0])
# ax2 = fig.add_subplot(gs[1, 0])
# ax3 = fig.add_subplot(gs[2, 0])

# plot_bar_on_broken_axes(
#     budgets,
#     rs,
#     [ax1, ax2, ax3],
#     top_ylim=[1e-4, 2e-3],
#     middle_ylim=[-1e-4, 1e-4],
#     bottom_ylim=[-0.1, -1e-4],
#     ylabel='Total moisture budget\n(kg m$^{-2}$ s$-1$)',
#     categories=WATER_VARS.values(),
#     text_kwargs=dict(x=-0.2, y=1.2e-3, s='B', fontsize=10, fontweight='bold'),
# )

gs = fig.add_gridspec(
    1,
    1,
    top=bottom + 2 * height + hspace,
    bottom=bottom + height + hspace,
    left=left + width + wspace,
    right=left + 3 * width + 2 * wspace,
)
ax = fig.add_subplot(gs[0, 0])

budgets_plot = {}
categories = []
for sim, b in budgets.items():
  buf = {}
  for l, v in b.items():
    if l == '$\mathcal{M}_{evap}$':
      buf[l] = 1e2 * v
      if sim == 'u1_mc0':
        categories.append(l + '\n($\\times 10^2$)')
    elif l in ('$\mathcal{M}_{comb}$', '$\mathcal{M}_{ma-exch}$'):
      buf[l] = 1e1 * v
      if sim == 'u1_mc0':
        categories.append(l + '\n($\\times 10$)')
    else:
      buf[l] = v
      if sim == 'u1_mc0':
        categories.append(l)
  budgets_plot[sim] = buf

plot_bar(
    budgets_plot,
    rs,
    ax,
    ylim=[-5e-2, 2e-2],
    ylabel='Total humidity budget\n(kg m$^{-2}$ s$^{-1}$)',
    categories=categories,
    text_kwargs=dict(x=-0.2, y=1.1e-2, s='B', fontsize=10, fontweight='bold'),
    bar_width=bar_width,
    yticks=[-4e-2, -2e-2, 0, 2e-2],
)


# The energy budget.
budgets = {}
for sim in labels.keys():
  buf = {}
  for v, l in ENERGY_VARS.items():
    buf[l] = (
        parcel_plot.sel(sim=sim, var=v)
        .sel(z=slice(0, 10e3))['mean']
        .integrate(coord='z')
        / 1e3
    )
  budgets[sim] = buf

rs = [np.arange(len(ENERGY_VARS))]
for i in range(1, len(labels)):
  rs.append(rs[i - 1] + bar_width)

# gs = fig.add_gridspec(
#     3,
#     1,
#     height_ratios=[1, 1, 1],
#     top=bottom + height,
#     bottom=bottom,
#     left=left,
#     right=left + 3 * width + 2 * wspace,
#     hspace=0.06,
# )
# ax1 = fig.add_subplot(gs[0, 0])
# ax2 = fig.add_subplot(gs[1, 0])
# ax3 = fig.add_subplot(gs[2, 0])

# plot_bar_on_broken_axes(
#     budgets,
#     rs,
#     [ax1, ax2, ax3],
#     top_ylim=[10, 150],
#     middle_ylim=[-0.1, 0.1],
#     bottom_ylim=[-100, -10],
#     ylabel='Total energy budget\n(kJ m$^{-2}$ s$-1$)',
#     categories=ENERGY_VARS.values(),
#     text_kwargs=dict(x=-0.2, y=100, s='C', fontsize=10, fontweight='bold'),
# )

gs = fig.add_gridspec(
    1,
    1,
    top=bottom + height,
    bottom=bottom,
    left=left,
    right=left + 3 * width + 2 * wspace,
)
ax = fig.add_subplot(gs[0, 0])

budgets_plot = {}
categories = []
for sim, b in budgets.items():
  buf = {}
  for l, v in b.items():
    if l == '$\mathcal{E}_{evap}$':
      buf[l] = 1e2 * v
      if sim == 'u1_mc0':
        categories.append(l + '\n($\\times 10^2$)')
    elif l == '$\mathcal{W}_d$':
      buf[l] = 1e3 * v
      if sim == 'u1_mc0':
        categories.append(l + '\n($\\times 10^3$)')
    else:
      buf[l] = v
      if sim == 'u1_mc0':
        categories.append(l)
  budgets_plot[sim] = buf

plot_bar(
    budgets_plot,
    rs,
    ax,
    ylim=[-1e2, 2e2],
    ylabel='Total energy budget\n(kJ m$^{-2}$ s$^{-1}$)',
    categories=categories,
    text_kwargs=dict(x=-0.2, y=1.6e2, s='C', fontsize=10, fontweight='bold'),
    bar_width=bar_width,
    yticks=[-1e2, 0, 1e2, 2e2],
)

if write_to_disk:
  filename = f'{plot_path}/fig_3.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
#@title Figure 3 (mass flux only)
def prepare_data(data_path):
  filepaths = {
      'Baseline': f'{data_path}/u1_mc0/postprocess',
      '2/3 Wind': f'{data_path}/u0.666_mc0/postprocess',
      '1/3 Wind': f'{data_path}/u0.333_mc0/postprocess',
  }

  sims = {}
  for k, filepath in filepaths.items():
    buf = xarray.open_zarr(
        f'{filepath}/flux_and_entrainment.zarr',
        decode_timedelta=True,
    )
    sims[k] = buf.isel(t=~buf.t.isnull())

  t_min = 20  # minutes
  z_max = 1.5e4  # meter
  return {
      k: (
          agg_ds.flux.sel(
              t=slice(np.timedelta64(t_min, 'm'), None), z=slice(0, z_max)
          )
          .mean(['t'])
          .compute() * 1e-6  # to units of t/hr
      )
      for k, agg_ds in sims.items()
  }


flux_mean = prepare_data(data_path)


fig, ax1 = plt.subplots(figsize=(4, 2.7))
for i, (k, f) in enumerate(flux_mean.items()):
  ax1.plot(f, f.z / 1e3, color=colors[i], label=f'Case {i + 1}')

ax1.set_xlabel('Mass Flux (kt s$^{-1}$)', fontsize=10)
ax1.set_xlim([0, 400])
ax1.xaxis.get_offset_text().set_fontsize(7)
ax1.tick_params(axis='x', labelsize=10)

ax1.set_ylabel('z (km)', fontsize=10)
ax1.set_ylim([0, 15])
ax1.tick_params(axis='y', labelsize=10)
ax1.set_yticks([0, 5, 10, 15])

ax1.legend(fontsize=10)

filename = f'{plot_path}/fig_3_mass_flux.png'
with gfile.Open(filename, 'wb') as f:
  plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
print(filename)

In [ ]:
# @title Figure 4

fig = plt.figure(figsize=(7.24, 7.24), constrained_layout=False)
gs1 = fig.add_gridspec(
    3,
    2,
    bottom=0.6,
    top=0.98,
    left=0.02,
    right=0.58,
    height_ratios=[0.06, 0.47, 0.47],
    wspace=0.06,
    hspace=0.05,
)
gs2 = fig.add_gridspec(
    2, 2, bottom=0.02, top=0.54, left=0.02, right=0.58, wspace=0.05, hspace=0.06
)
gs3 = fig.add_gridspec(5, 1, bottom=0.02, top=0.98, left=0.68, right=0.98)

ax1 = fig.add_subplot(gs1[1, 0])
ax2 = fig.add_subplot(gs1[1, 1])
ax3 = fig.add_subplot(gs1[2, 0])
ax4 = fig.add_subplot(gs1[2, 1])
ax5 = fig.add_subplot(gs2[0, 0])
ax6 = fig.add_subplot(gs2[0, 1])
ax7 = fig.add_subplot(gs2[1, 0])
ax8 = fig.add_subplot(gs2[1, 1])
ax9 = fig.add_subplot(gs3[0, 0])
ax10 = fig.add_subplot(gs3[1, 0])
ax11 = fig.add_subplot(gs3[2, 0])
ax12 = fig.add_subplot(gs3[3, 0])
ax13 = fig.add_subplot(gs3[4, 0])

subfig_labels = (
    'A',
    'B',
    'C',
    'D',
    'E',
    'F',
    'G',
    'H',
    'I',
    'J',
    'K',
    'L',
    'M',
)

# Panel 1: The flow field contours.
axs = [ax1, ax2, ax3, ax4]
images = []
for i, (ax, sim_name) in enumerate(
    zip(axs, ['u1_mc0', 'u1_mc0.3', 'u0.333_mc0', 'u0.666_mc0'])
):
  c = plot_xy(
      ax,
      subfig_labels[i],
      sim_name=sim_name,
      t=52,
      height=12,
      show_colorbar=False,
      show_xlabel=ax in (ax3, ax4),
      show_ylabel=ax in (ax1, ax3),
  )
  images.append(c)
cax = fig.add_subplot(gs1[0, :])
cb = plt.colorbar(
    images[0],
    cax=cax,
    orientation='horizontal',
    location='top',
    fraction=0.046,
    pad=0.04,
    extend='both',
)
cb.set_label(r'$|\boldsymbol{u}|$ (m/s)', size=10)
cb.ax.tick_params(labelsize=10)


# Panel 2: the isochrone contours.
t = 30  # In units of minutes.
nt = 4
dt = 30
x0 = 0
x1 = 15
y0 = -8
y1 = 8


for i, ax in enumerate([ax5, ax6, ax7, ax8]):
  for j, (label, ds) in enumerate(sims.items()):
    ds = ds.isel(t=~ds.t.isnull())
    ds = ds.assign_coords(x=ds.x / 1e3, y=ds.y / 1e3, z=ds.z / 1e3)
    t_s = (
        ds.T_s.sel(
            z=12e-3,
            method='nearest',
        )
        .sel(
            t=np.timedelta64(t, 'm'),
        )
        .sel(x=slice(x0, x1), y=slice(y0, y1))
        .compute()
    )
    c_fire = ax.contour(
        t_s.x,
        t_s.y,
        t_s.T,
        levels=[600.0],
        colors=[colors[j]],
        linewidths=0.5,
    )
    q_r = (
        ds.q_r.sel(
            z=12e-3,
            method='nearest',
        )
        .sel(
            t=np.timedelta64(t, 'm'),
        )
        .sel(x=slice(x0, x1), y=slice(y0, y1))
    )
    c_rain = ax.contour(
        q_r.x,
        q_r.y,
        q_r.T,
        levels=[1e-5],
        colors=colors[j],
        linestyles='dashed',
        linewidths=0.5,
        alpha=0.6,
    )
  ax.set_aspect('equal')

  ax.set_xlim([x0, x1])
  if i in (2, 3):
    ax.set_xlabel('$x$ (km)', fontsize=10)
    ax.tick_params(axis='x', labelsize=10)
  else:
    ax.set_xticklabels([])

  ax.set_ylim([y0, y1])
  ax.set_yticks([-8, -4, 0, 4, 8])
  if i in (0, 2):
    ax.set_ylabel('$y$ (km)', fontsize=10)
    ax.tick_params(axis='y', labelsize=10)
  else:
    ax.set_yticklabels([])

  ax.text(1, 6, subfig_labels[4 + i], fontsize=10, fontweight='bold')
  ax.text(2.5, 6, f'{int(t):d} min', fontsize=10)

  t += dt

# Panel 3: fire and cloud stats.
## Fire power and parcel fractions.
power = {
    'Baseline': xarray.open_zarr(
        f'{data_path}/u1_mc0/postprocess/power.zarr/',
        decode_timedelta=True,
    ),
    '1/3 Wind': xarray.open_zarr(
        f'{data_path}/u0.333_mc0/postprocess/power.zarr/',
        decode_timedelta=True,
    ),
    '2/3 Wind': xarray.open_zarr(
        f'{data_path}/u0.666_mc0/postprocess/power.zarr/',
        decode_timedelta=True,
    ),
    '30% MC': xarray.open_zarr(
        f'{data_path}/u1_mc0.3/postprocess/power.zarr/',
        decode_timedelta=True,
    ),
}
summary = summarize_masks(parcels)
recirc = {k: ds.isel(mask=2) / 1e5 for k, ds in summary.items()}

for i, (label, ds) in enumerate(power.items()):
  ax9.plot(
      ds.t / np.timedelta64(1, 'm'), ds.fire_power, label=label, color=colors[i]
  )
  print(
      f'{label}:'
      f' {ds.fire_power.sel(t=slice(np.timedelta64(20, "m"), None)).mean("t").compute()} GW'
  )
# ax.plot([0, 90], [101, 101], 'k-.')

ax9_2 = ax9.twinx()

for i, (k, da) in enumerate(recirc.items()):
  ax9_2.plot(
      da.t / np.timedelta64(1, 'm'),
      da * 100,
      label=labels[k],
      linestyle='--',
      color=colors[i],
  )
  print(
      f'{k}:'
      f' {da.sel(t=slice(np.timedelta64(20, "m"), None)).mean("t").compute() * 100} %'
  )

# ax9.set_xlabel('t (min)', fontsize=10)
# ax9.set_xlim([0, 90])
# ax9.set_xticks([0, 15, 30, 45, 60, 75, 90])
# ax9.tick_params(axis='x', labelsize=10)
ax9.set_xlim([0, 120])
ax9.set_xticks([0, 30, 60, 90, 120])
ax9.set_xticklabels([])

ax9.set_ylabel('Fire power (GW)', fontsize=10)
ax9.set_yscale('log')
ax9.set_ylim([10, 1e4])
# ax9.set_ylim([0, 2.5e3])
ax9.tick_params(axis='y', labelsize=10)

ax9_2.set_ylabel('Parcel fraction (%)', fontsize=10)
# ax2.set_ylim([1e-1, 1e2])
ax9_2.set_ylim([0, 6])
ax9_2.tick_params(axis='y', labelsize=10)

ax9.text(10, 3e3, subfig_labels[8], fontsize=10, fontweight='bold')

## Mass flux.
Z = 1e3

filepaths = {
    'Baseline': f'{data_path}/u1_mc0/postprocess',
    '1/3 Wind': f'{data_path}/u0.333_mc0/postprocess',
    '2/3 Wind': f'{data_path}/u0.666_mc0/postprocess',
    '30% MC': f'{data_path}/u1_mc0.3/postprocess',
}
flux = {}
for k, filepath in filepaths.items():
  ds = xarray.open_zarr(
      f'{filepath}/flux_and_entrainment.zarr',
      decode_timedelta=True,
  )
  ds = ds.isel(t=~ds.t.isnull())
  flux[k] = ds.flux.sel(z=Z, method='nearest')


for i, (k, f) in enumerate(flux.items()):
  ax10.plot(f.t / np.timedelta64(1, 'm'), f, label=k, color=colors[i])

# ax10.set_xlabel('t (min)', fontsize=10)
# ax10.tick_params(axis='x', labelsize=10)
ax10.set_xlim([0, 120])
ax10.set_xticks([0, 30, 60, 90, 120])
ax10.set_xticklabels([])

ax10.set_ylabel('Mass Flux (kg s$^{-1}$)', fontsize=10)
ax10.set_ylim([0, 6e8])
ax10.yaxis.get_offset_text().set_fontsize(7)
ax10.tick_params(axis='y', labelsize=10)

ax10.text(10, 4.5e8, subfig_labels[9], fontsize=10, fontweight='bold')

## Total precipitation.
sim_names = ['u1_mc0', 'u0.333_mc0', 'u0.666_mc0', 'u1_mc0.3']
q_vars = {'q_r': 'Rain', 'q_s': 'Snow', 'q_l': 'Cloud(l)', 'q_i': 'Cloud(i)'}
linestyles = ['-', '--', ':', '-.']

for i, sim_name in enumerate(sim_names):
  ds = xarray.open_zarr(
      f'{data_path}/{sim_name}/postprocess/water.zarr/',
      decode_timedelta=True,
  )
  ds = ds.isel(t=~ds.t.isnull())
  for j, (varname, label) in enumerate(q_vars.items()):
    if varname not in ('q_r', 'q_s'):
      continue
    ax11.semilogy(
        ds.t / np.timedelta64(1, 'm'),
        ds[varname].sum('z') * 1e-3 * 1.000439,
        linestyle=linestyles[j],
        color=colors[i],
    )

# ax11.set_xlabel('t (min)', fontsize=10)
# ax11.tick_params(axis='x', labelsize=10)
ax11.set_xlim([0, 120])
ax11.set_xticks([0, 30, 60, 90, 120])
ax11.set_xticklabels([])

ax11.set_ylabel('Total precipitation (t)', fontsize=10)
ax11.set_ylim([1e1, 1e7])
ax11.tick_params(axis='y', labelsize=10)
# ax11.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax11.yaxis.get_offset_text().set_fontsize(7)

ax11.text(10, 1e6, subfig_labels[10], fontsize=10, fontweight='bold')

## Burn area (vs. t).
filepaths = {
    'Baseline': f'{data_path}/u1_mc0/postprocess',
    '1/3 Wind': f'{data_path}/u0.333_mc0/postprocess',
    '2/3 Wind': f'{data_path}/u0.666_mc0/postprocess',
    '30% MC': f'{data_path}/u1_mc0.3/postprocess',
}
ba = {}
br = {}
for k, filepath in filepaths.items():
  ds = xarray.open_zarr(
      f'{filepath}/burn_area.zarr',
      decode_timedelta=True,
  )
  ds = ds.isel(t=~ds.t.isnull())
  ba[k] = ds.burn_area * 1e-6
  br[k] = ba[k].diff('t', label='lower') / (
      ba[k].t.diff('t', label='lower') / np.timedelta64(1, 's')
  )

for i, (k, f) in enumerate(ba.items()):
  ax12.plot(f.t / np.timedelta64(1, 'm'), f, label=k, color=colors[i])

ax12.set_ylabel('Burn area (km$^{2}$)', fontsize=10)
ax12.set_ylim([0, 80])
ax12.set_yticks([0, 20, 40, 60, 80])
ax12.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax12.yaxis.get_offset_text().set_fontsize(7)
ax12.tick_params(axis='y', labelsize=10)

# ax12.set_xlabel('t (min)', fontsize=10)
# ax12.tick_params(axis='x', labelsize=10)
ax12.set_xlim([0, 120])
ax12.set_xticks([0, 30, 60, 90, 120])
ax12.set_xticklabels([])

ax12.text(10, 65, subfig_labels[11], fontsize=10, fontweight='bold')

## Burn rate.
for i, (k, f) in enumerate(br.items()):
  ax13.plot(f.t / np.timedelta64(1, 'm'), f, label=k, color=colors[i])

ax13.set_ylabel('Burn rate (km$^{2}$ s$^{-1}$)', fontsize=10)
ax13.set_yscale('log')
ax13.set_ylim([1e-4, 1e-1])
# ax13.set_ylim([0, 0.06])
# ax13.set_yticks([0, 0.02, 0.04, 0.06])
# ax13.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax13.yaxis.get_offset_text().set_fontsize(7)
ax13.tick_params(axis='y', labelsize=10)

ax13.set_xlabel('t (min)', fontsize=10)
ax13.set_xlim([0, 120])
ax13.set_xticks([0, 30, 60, 90, 120])
ax13.tick_params(axis='x', labelsize=10)

ax13.text(10, 3e-2, subfig_labels[12], fontsize=10, fontweight='bold')

if write_to_disk:
  filename = f'{plot_path}/fig_4.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
#@title Figure S2
ds = sims['Baseline']
u = ds.u.isel(t=0)
x = u.x.compute() * 1e-3
z = u.z.compute() * 1e-3

fig = plt.figure(figsize=(4.76, 1.6))

# Instantaneous snapshot.
gs = fig.add_gridspec(
    1,
    1,
    bottom=0.02,
    top=0.98,
    left=0.02,
    right=0.52,
)
ax1 = fig.add_subplot(gs[0, 0])
c = ax1.pcolormesh(
    x,
    z,
    u.sel(y=0, method='nearest').T.compute(),
    cmap=div,
)

cb = plt.colorbar(
    c,
    ax=ax1,
    orientation='horizontal',
    location='top',
    fraction=0.06,
    pad=0.04,
    extend='both',
)
cb.set_label(r'$u$ (m/s)', size=10)
cb.ax.tick_params(labelsize=10)

ax1.set_aspect('equal')
ax1.set_xlabel('x (km)', fontsize=10)
ax1.tick_params(axis='x', labelsize=10)

ax1.set_ylabel('z (km)', fontsize=10)
ax1.set_yticks([0, 5, 10, 15, 20])
ax1.tick_params(axis='y', labelsize=10)

ax1.text(2.7, 16, 'A', fontsize=10, fontweight='bold')

# Mean profile.
gs = fig.add_gridspec(
    1,
    1,
    bottom=0.02,
    top=0.88,
    left=0.54,
    right=0.98,
)
ax2 = fig.add_subplot(gs[0, 0])

u_mean = u.mean(dim=['x', 'y']).compute()

ax2.plot(u_mean, z, label='Baseline')

ax2.set_xlabel('u (m/s)', fontsize=10)
ax2.set_xlim([0, 35])
ax2.tick_params(axis='x', labelsize=10)

ax2.set_ylim([0, 20])
ax2.set_yticklabels([])

ax2.text(3, 16, 'B', fontsize=10, fontweight='bold')

# fig.set_layout_engine('tight')
if write_to_disk:
  filename = f'{plot_path}/fig_s2.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
#@title Figure 3

sim_labels = ['u1_mc0', 'u0.333_mc0', 'u0.666_mc0', 'u1_mc0.3']
times = ['30.0', '60.0', '90.0', '120.0']

fig = plt.figure(figsize=(7.24, 5), constrained_layout=False)
gs = fig.add_gridspec(
    len(sim_labels),
    len(times),
    bottom=0,
    top=1,
    left=0,
    right=1,
    hspace=0.02,
    wspace=0.02,
)

for r, sim in enumerate(sim_labels):
  for c, t in enumerate(times):
    if sim == 'u1_mc0' and t == '120.0':
      filename = f'{plot_path}/demo/pyrocb_demo.png'
    else:
      filename = f'{data_path}/3d_render/{sim}_{t}.png'
    pyrocb = imread(filename)

    ax = fig.add_subplot(gs[r, c])

    ax.imshow(pyrocb[100:-40, 125:-300, :])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
      spine.set_edgecolor('none')
    ax.text(
        20,
        80,
        chr(ord('A') + r * len(times) + c),
        fontsize=10,
        fontweight='bold',
    )

if write_to_disk:
  filename = f'{plot_path}/fig_s3.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure S4
mask_labels = {
    'Fire': 'plume',
    'Anvil': 'aloft',
    'Dispersion': 'dispersed_on_ground',
    'Feedback': 'feedback',
}


summary = summarize_masks(parcels)
parcel_frac = {
    k: {label: ds.sel(mask=mask) / 1e5 for label, mask in mask_labels.items()}
    for k, ds in summary.items()
}

bottom = 0.02
left = 0.02
height = 0.46
width = 0.46
hspace = 0.06
wspace = 0.06

fig = plt.figure(figsize=(4.76, 2.38), constrained_layout=False)

for i, (sim, das_pf) in enumerate(parcel_frac.items()):
  r = i // 2
  c = i % 2
  gs = fig.add_gridspec(
      2,
      1,
      bottom=bottom + (1 - r) * (height + hspace),
      top=bottom + (1 - r) * (height + hspace) + height,
      left=left + c * (width + wspace),
      right=left + c * (width + wspace) + width,
      height_ratios=[1, 1],
      hspace=0.04,
  )
  ax1 = fig.add_subplot(gs[0, 0])
  ax2 = fig.add_subplot(gs[1, 0])

  for j, (label, da_pf) in enumerate(das_pf.items()):
    ax_id = 0 if label in ('Feedback', 'Dispersion') else 1
    plot_on_brokenaxes(
        da_pf.t / np.timedelta64(1, 'm'),
        da_pf * 100,
        ax1,
        ax2,
        xlim=[0, 90],
        ylim_top=[6, 30],
        ylim_bottom=[0, 6],
        yticks_top=[10, 20, 30],
        yticks_bottom=[0, 3, 6],
        xlabel='t (min)' if r == 1 else None,
        ylabel=None,
        show_xlabel=r == 1,
        show_ylabel=c == 0,
        xaxis_sci=False,
        color=colors[j],
        fontsize=10,
        text_kwargs=dict(
            x=4, y=22, s=chr(ord('A') + i), fontsize=10, fontweight='bold'
        ),
    )
    if sim == 'u1_mc0':
      ax1.plot([82, 82], [0, 1e2], 'k--')
      ax2.plot([82, 82], [0, 1e2], 'k--')
    stats = (
        da_pf.sel(
            t=[
                np.timedelta64(27, 'm'),
                np.timedelta64(52, 'm'),
                np.timedelta64(82, 'm'),
            ]
        )
        .compute()
        .data
    )
    print(f'{sim} {label}: {stats[0]:.1%} {stats[1]:.1%} {stats[2]:.1%}')

gs = fig.add_gridspec(
    1,
    1,
    bottom=bottom,
    top=bottom + 2 * height + hspace,
    left=0,
    right=left,
)
ax = fig.add_subplot(gs[0, 0], frameon=False)
ax.tick_params(
    labelcolor='none', top=False, bottom=False, left=False, right=False
)  # Hide ticks and labels

# Set the common y-axis label
ax.set_ylabel('Parcel fraction (%)', rotation='vertical', fontsize=10)


if write_to_disk:
  filename = f'{plot_path}/fig_s4.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
#@title Figure S6
fig = plt.figure(figsize=(7.24, 6), constrained_layout=False)

sim_names = ['u1_mc0', 'u0.333_mc0', 'u0.666_mc0', 'u1_mc0.3']
times = [27, 52, 82]

gs = fig.add_gridspec(
    len(sim_names),
    len(times) + 1,
    bottom=0.02,
    top=0.98,
    left=0.02,
    right=0.98,
    hspace=0.06,
    wspace=0.06,
    width_ratios=[1, 1, 1, 0.1],
)

image = None
for r, sim_name in enumerate(sim_names):
  for c, t in enumerate(times):
    ax = fig.add_subplot(gs[r, c])

    image = plot_xy(
        ax,
        chr(ord('A') + r * 3 + c),
        sim_name=sim_name,
        t=t,
        height=12,
        show_colorbar=False,
        show_xlabel=r == 3,
        show_ylabel=c == 0,
    )

cax = fig.add_subplot(gs[:, -1])
cb = plt.colorbar(
    image,
    cax=cax,
    orientation='vertical',  # 'horizontal',
    location='right',  # 'top',
    fraction=0.046,
    pad=0.04,
    extend='both',
)
cb.set_label(r'$|\boldsymbol{u}|$ (m/s)', size=10)
cb.ax.tick_params(labelsize=10)

if write_to_disk:
  filename = f'{plot_path}/fig_s6.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure S7


def add_sum(pds):
  src_sum = 0
  for v in pds['var'].data:
    if v.startswith('energy_budget'):
      src_sum += pds.sel(var=v)
  rad_total = pds.sel(var='energy_budget_radiation') + pds.sel(
      var='energy_budget_solid_radiation'
  )
  return xarray.concat(
      [
          pds,
          src_sum.assign_coords(var='energy_budget_total'),
          rad_total.assign_coords(var='energy_budget_total_radiation'),
      ],
      dim='var',
  )


parcel_plot = (
    parcels_budget.sel(w_dir='up', mask='plume')
    # .sel(start_min=slice(np.timedelta64(35, 'm'), np.timedelta64(45, 'm')))
    .sel(t=slice(np.timedelta64(35, 'm'), None)).mean('t')
    / 1e3
)
parcel_plot = add_sum(parcel_plot)

bottom = 0.02
left = 0.02
height = 0.41
width = 0.3
hspace = 0.16
wspace = 0.03

fig = plt.figure(figsize=(7.24, 3.6), constrained_layout=False)

axes = []
for r in range(3):
  for c in range(3):
    if r == 2 and c > 0:
      break
    gs = fig.add_gridspec(
        2,
        1,
        bottom=bottom + (1 - r) * (height + hspace),
        top=bottom + (1 - r) * (height + hspace) + height,
        left=left + c * (width + wspace),
        right=left + c * (width + wspace) + width,
        height_ratios=[1, 1],
        hspace=0.04,
    )
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    axes.append([ax1, ax2])

# Potential energy.
v = 'energy_budget_potential_energy'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[0][0],
      axes[0][1],
      xlim=[-2e-2, 1e-2],
      ylim_top=[0.5, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-2, y=9, s='A', fontsize=10, fontweight='bold')
  )

# Drag heating.
v = 'energy_budget_drag_heating'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[1][0],
      axes[1][1],
      xlim=[-1e-3, 4e-3],
      ylim_top=[0.5, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-0.8e-3, y=9, s='B', fontsize=10, fontweight='bold')
  )

# Radiation heat flux.
v = 'energy_budget_radiation'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[2][0],
      axes[2][1],
      xlim=[-6, 1],
      ylim_top=[0.5, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-5.8, y=9, s='C', fontsize=10, fontweight='bold')
  )

# Evaporation energy.
v = 'energy_budget_latent_heat_fuel_moisture'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[3][0],
      axes[3][1],
      xlim=[-2e-1, 1e-1],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-1, y=9, s='D', fontsize=10, fontweight='bold')
  )

# Combustion energy.
v = 'energy_budget_combustion'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[4][0],
      axes[4][1],
      xlim=[-10, 20],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-8, y=9, s='E', fontsize=10, fontweight='bold')
  )

# Latent heat from precipitation.
v = 'energy_budget_latent_heat_microphysics'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[5][0],
      axes[5][1],
      xlim=[-1e-2, 5e-2],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-0.8e-2, y=9, s='F', fontsize=10, fontweight='bold')
  )

# Heat from mass exchange.
v = 'energy_budget_mass_source'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[6][0],
      axes[6][1],
      xlim=[-5e-1, 1e-1],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{ENERGY_VARS[v]} (kJ m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-4.8e-1, y=9, s='G', fontsize=10, fontweight='bold')
  )

if write_to_disk:
  filename = f'{plot_path}/fig_s7.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure S8


def add_sum(pds):
  src_sum = 0
  for v in pds['var'].data:
    if v.startswith('energy_budget'):
      src_sum += pds.sel(var=v)
  rad_total = pds.sel(var='energy_budget_radiation') + pds.sel(
      var='energy_budget_solid_radiation'
  )
  return xarray.concat(
      [
          pds,
          src_sum.assign_coords(var='energy_budget_total'),
          rad_total.assign_coords(var='energy_budget_total_radiation'),
      ],
      dim='var',
  )


parcel_plot = (
    parcels_budget.sel(w_dir='up', mask='plume')
    # .sel(start_min=slice(np.timedelta64(35, 'm'), np.timedelta64(45, 'm')))
    .sel(t=slice(np.timedelta64(35, 'm'), None)).mean('t')
)
parcel_plot = add_sum(parcel_plot)

bottom = 0.02
left = 0.02
height = 0.41
width = 0.3
hspace = 0.16
wspace = 0.03

fig = plt.figure(figsize=(7.24, 3.6), constrained_layout=False)

axes = []
for r in range(2):
  for c in range(3):
    if r == 1 and c > 1:
      break
    gs = fig.add_gridspec(
        2,
        1,
        bottom=bottom + (1 - r) * (height + hspace),
        top=bottom + (1 - r) * (height + hspace) + height,
        left=left + c * (width + wspace),
        right=left + c * (width + wspace) + width,
        height_ratios=[1, 1],
        hspace=0.04,
    )
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    axes.append([ax1, ax2])

# Rain.
v = 'water_budget_rain'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[0][0],
      axes[0][1],
      xlim=[-2e-5, 1e-5],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-5, y=9, s='A', fontsize=10, fontweight='bold')
  )

# Snow.
v = 'water_budget_snow'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[1][0],
      axes[1][1],
      xlim=[-2e-5, 1e-5],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-5, y=9, s='B', fontsize=10, fontweight='bold')
  )

# Evaporation moisture.
v = 'water_budget_dehydration'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[2][0],
      axes[2][1],
      xlim=[-1e-5, 4e-5],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-0.8e-5, y=9, s='C', fontsize=10, fontweight='bold')
  )

# Combustion moisture.
v = 'water_budget_combustion'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[3][0],
      axes[3][1],
      xlim=[-1e-4, 4e-4],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-0.8e-4, y=9, s='D', fontsize=10, fontweight='bold')
  )

# Mass exchange.
v = 'water_budget_mass_source'
for i, sim in enumerate(labels.keys()):
  pds_plot = parcel_plot.sel(sim=sim)
  plot_on_brokenaxes(
      pds_plot.sel(var=v)['mean'],
      pds_plot.z * 1e-3,
      axes[4][0],
      axes[4][1],
      xlim=[-2e-5, 1e-5],
      ylim_top=[1, 12],
      ylim_bottom=[0, 0.05],
      yticks_top=[4, 8, 12],
      yticks_bottom=[0, 0.025, 0.05],
      xlabel=f'{WATER_VARS[v]} (kg m$^{{-3}}$ s$^{{-1}}$)',
      # ylabel='z (km)',
      color=colors[i],
      fontsize=10,
      text_kwargs=dict(x=-1.8e-5, y=9, s='E', fontsize=10, fontweight='bold')
  )

if write_to_disk:
  filename = f'{plot_path}/fig_s8.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)

In [ ]:
# @title Figure S9
sims = {
    'Baseline': xarray.open_zarr(
        f'{data_path}/u1_mc0/postprocess/fire_front.zarr',
        decode_timedelta=True,
    ),
    '1/3 Wind': xarray.open_zarr(
        f'{data_path}/u0.333_mc0/postprocess/fire_front.zarr',
        decode_timedelta=True,
    ),
    '2/3 Wind': xarray.open_zarr(
        f'{data_path}/u0.666_mc0/postprocess/fire_front.zarr',
        decode_timedelta=True,
    ),
    '30% MC': xarray.open_zarr(
        f'{data_path}/u1_mc0.3/postprocess/fire_front.zarr',
        decode_timedelta=True,
    ),
}

fire_locs = ['front', 'back', 'left']
stride = 10
t0 = np.timedelta64(10, 'm')

fig = plt.figure(figsize=(7.24, 2.4), constrained_layout=False)
gs = fig.add_gridspec(
    2,
    3,
    bottom=0,
    top=1,
    left=0,
    right=1,
    hspace=0.12,
    wspace=0.22,
)

for c, fire_loc in enumerate(fire_locs):
  ff = {}
  ros = {}
  for k, ds in sims.items():
    ds = ds.isel(t=~ds.t.isnull())
    ff[k] = ds[fire_loc]

    # Compute the ROS.
    ros_k = (
        (
            ff[k].diff('t', label='lower')
            / (ff[k].t.diff('t', label='lower') / np.timedelta64(1, 's'))
        )
        .sel(t=slice(t0, None))
        .compute()
    )
    n = len(ros_k) // stride
    t = ros_k.t / np.timedelta64(1, 's')
    t_filtered = t[1 : 1 + n * stride : stride]
    ros_filtered = np.zeros((n,), dtype=np.float32)
    for j in range(stride):
      ros_filtered += ros_k[j : n * stride : stride].data
    ros_filtered /= stride
    ros[k] = xarray.DataArray(
        data=np.abs(ros_filtered),
        dims=['t'],
        coords=dict(t=t_filtered * np.timedelta64(1, 's')),
    )


  ax1 = fig.add_subplot(gs[0, c])
  ax2 = fig.add_subplot(gs[1, c])

  # Plot the fire front location.
  for i, (label, da) in enumerate(ff.items()):
    t = da.t / np.timedelta64(1, 'm')
    ax1.plot(t, da * 1e-3, label=label, color=colors[i])

  ax1.set_xlim([0, 120])
  ax1.set_xticks([0, 30, 60, 90, 120])
  ax1.set_xticklabels([])

  ax1.set_ylabel('Fire front (km)', fontsize=10)
  if fire_loc == 'left':
    ax1.set_ylim([0, 6])
  elif fire_loc == 'front':
    ax1.set_ylim([4, 10])
  elif fire_loc == 'back':
    ax1.set_ylim([0, 6])
  ax1.tick_params(axis='y', labelsize=10)

  ax1.text(6, 9 if fire_loc == 'front' else 5, chr(ord('A') + c), fontsize=10, fontweight='bold')

  # Plot the rate of spread.
  for i, (label, da) in enumerate(ros.items()):
    t = da.t / np.timedelta64(1, 'm')
    ax2.plot(t, da, label=label, color=colors[i])

  ax2.set_xlabel('t (min)', fontsize=10)
  ax2.set_xlim([0, 120])
  ax2.set_xticks([0, 30, 60, 90, 120])
  ax2.tick_params(axis='x', labelsize=10)

  ax2.set_ylabel('ROS (m s$^{-1}$)', fontsize=10)
  ax2.set_ylim([0, 3])
  ax2.set_yticks([0, 1, 2, 3])
  ax2.tick_params(axis='y', labelsize=10)

  ax2.text(6, 2.5, chr(ord('A') + c + 3), fontsize=10, fontweight='bold')

if write_to_disk:
  filename = f'{plot_path}/fig_s9.png'
  with gfile.Open(filename, 'wb') as f:
    plt.savefig(f, format='png', dpi=300, bbox_inches='tight')
  print(filename)
